In [8]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
df = pd.read_csv("../data/raw/train.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store", "item", "date"])


In [3]:
def create_sequences(series, window_size=30):
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i+window_size])
        y.append(series[i+window_size])
    return np.array(X), np.array(y)


In [4]:
scaler = MinMaxScaler()

df["sales_scaled"] = scaler.fit_transform(df[["sales"]])

WINDOW = 60

X_all, y_all = [], []

for (store, item), group in df.groupby(["store", "item"]):
    sales = group["sales_scaled"].values.reshape(-1, 1)

    if len(sales) <= WINDOW:
        continue

    X, y = create_sequences(sales, WINDOW)

    X_all.append(X)
    y_all.append(y)

X_all = np.vstack(X_all)
y_all = np.vstack(y_all)

print(X_all.shape, y_all.shape)


(883000, 60, 1) (883000, 1)


In [5]:
split = int(0.8 * len(X_all))

X_train, X_val = X_all[:split], X_all[split:]
y_train, y_val = y_all[:split], y_all[split:]


In [9]:
model = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1)
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 60, 128)             │          66,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 60, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 116,033 (453.25 KB)

 Trainable params: 116,033 (453.25 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 966s 87ms/step - loss: 0.0019 - val_loss: 0.0014
Epoch 2/50
 8395/11038 ━━━━━━━━━━━━━━━━━━━━ 3:05 70ms/step - loss: 0.0015

KeyboardInterrupt: 

In [11]:
y_pred = model.predict(X_val)

# Inverse scaling
y_val_inv = scaler.inverse_transform(y_val)
y_pred_inv = scaler.inverse_transform(y_pred)

mae = mean_absolute_error(y_val_inv, y_pred_inv)
rmse = root_mean_squared_error(y_val_inv, y_pred_inv)
mape = np.mean(np.abs((y_val_inv - y_pred_inv) / y_val_inv)) * 100

print(f"LSTM MAE  : {mae:.2f}")
print(f"LSTM RMSE : {rmse:.2f}")
print(f"LSTM MAPE : {mape:.2f}%")


5613/5613 ━━━━━━━━━━━━━━━━━━━━ 33s 6ms/step
LSTM MAE  : 8.92
LSTM RMSE : 11.42
LSTM MAPE : 12.54%


In [13]:
model.save("../models/lstm_model.keras")
